# 2 · Limpieza y transformación del dataset

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Aplicar, de forma controlada y documentada, las decisiones identificadas en
`01_exploracion_inicial.ipynb`. Cada transformación va acompañada de su justificación
y de una comprobación posterior del resultado.

**Criterio general seguido:** ante un valor desconocido, se prefiere *representar la
ausencia* antes que *inventar el valor*. Solo se imputa cuando el número de registros
afectados es despreciable.

| | |
|---|---|
| **Entrada** | `data/processed/diabetic_data_replaceNaN.csv` (101.766 × 50) |
| **Salida** | `data/processed/diabetic_data_clean.csv` (101.766 × 35) |
| **Filas eliminadas** | Ninguna |
| **Columnas eliminadas** | 15 |

### Contenido

1. Configuración y carga
2. Eliminación de columnas sin valor analítico
3. Tratamiento de los valores ausentes
4. Variables casi constantes
5. Conversión de tipos
6. Verificación de categorías
7. Registros duplicados
8. Rangos de las variables numéricas
9. Exportación
10. Conclusiones

## 1 · Configuración y carga

Se parte del fichero generado por el notebook 01, en el que los `?` ya han sido
convertidos a `NaN`. El dataset original de `data/raw/` no se toca en ningún momento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path(r'D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1')
RAW = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

# low_memory=False evita el DtypeWarning de payer_code, cuyo tipo Pandas
# interpreta de forma distinta según el bloque del fichero que esté leyendo.
df = pd.read_csv(PROC / 'diabetic_data_replaceNaN.csv', low_memory=False)

print(f'Dimensiones de partida: {df.shape}')
df.head()

Dimensiones de partida: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 2 · Eliminación de columnas sin valor analítico

Se eliminan cinco columnas, cada una por un motivo distinto:

| Columna | Motivo |
|---|---|
| `encounter_id` | Identificador del ingreso. No es una característica clínica; usarlo como predictor introduciría patrones artificiales |
| `patient_nbr` | Identificador del paciente. Mismo motivo |
| `weight` | ~97% de valores desconocidos: información insuficiente para ser utilizable |
| `examide` | Constante: todos los registros valen `"No"` |
| `citoglipton` | Constante: todos los registros valen `"No"` |

> **Nota para la fase de modelado:** `patient_nbr` se elimina como *predictor*, pero
> la información que contiene sigue siendo necesaria. El dataset tiene 101.766
> ingresos de solo 71.518 pacientes distintos, y esa duplicidad habrá que tenerla en
> cuenta al separar entrenamiento y test para no incurrir en fuga de datos.

In [2]:
df = df.drop(columns=['encounter_id', 'patient_nbr', 'examide', 'citoglipton', 'weight'])

print(f'Dimensiones tras la eliminación: {df.shape}')

Dimensiones tras la eliminación: (101766, 45)


## 3 · Tratamiento de los valores ausentes

No existe una única estrategia correcta: depende de **cuánta información falta** y de
**qué significa que falte**. La estrategia aplicada a cada variable es la siguiente:

| Variable | % ausente | Estrategia | Razonamiento |
|---|---|---|---|
| `gender` | ~0,003% | Moda | 3 registros sobre 101.766: irrelevante para la distribución |
| `race` | ~2,2% | Categoría `Unknown` | La ausencia no es aleatoria: depende del hospital y del registro administrativo |
| `medical_specialty` | ~49% | Categoría `Unknown` | Imputar la moda en la mitad del dataset sería inventar información |
| `payer_code` | ~40% | Categoría `Unknown` | Mismo criterio |
| `max_glu_serum`, `A1Cresult` | ~95% / ~83% | Categoría `Not_Measured` | No son datos perdidos: el valor original significa "prueba no realizada" |
| `diag_1`, `diag_2`, `diag_3` | <1,5% | Conservar `NaN` | Asignar un diagnóstico artificial carece de sentido clínico |

### 3.1 · `gender`

Contiene únicamente **3 valores** no válidos (`Unknown/Invalid`, convertidos a `NaN`
en el notebook 01). Es la única variable donde imputar con la moda está justificado:
tres registros no pueden alterar la distribución de un dataset de más de cien mil.

La alternativa —eliminar esas filas— tampoco sería grave, pero perdería el resto de
información clínica de esos tres pacientes sin necesidad.

In [3]:
df['gender'] = df['gender'].fillna(df['gender'].mode().iloc[0])

df['gender'].value_counts(dropna=False)

gender
Female    54711
Male      47055
Name: count, dtype: int64

### 3.2 · `race`

Aproximadamente un **2,2%** de valores ausentes (unos 2.270 registros).

Pese a ser un porcentaje pequeño, **no se imputa con la moda**. Asignar `Caucasian` a
esos pacientes sería inventar un dato demográfico del que no se dispone. Además, la
ausencia de este campo **no es aleatoria**: depende del centro hospitalario y de su
proceso de registro administrativo, y esa no-aleatoriedad es en sí misma información
que podría estar relacionada con la readmisión.

Se aplica el mismo criterio que a `payer_code` y `medical_specialty`: la ausencia se
representa mediante una categoría propia.

In [4]:
df['race'] = df['race'].fillna('Unknown')

df['race'].value_counts(dropna=False)

race
Caucasian          76099
AfricanAmerican    19210
Unknown             2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

### 3.3 · `diag_1`, `diag_2` y `diag_3`

Presentan un porcentaje reducido de valores ausentes (21, 358 y 1.423 registros).

**No se imputan.** Son códigos de diagnóstico: asignar la moda equivaldría a atribuir
una enfermedad concreta a un paciente cuyo diagnóstico se desconoce. Los `NaN` se
conservan tal cual y su tratamiento se decidirá en la fase de preparación para el
modelado, junto con la agrupación de estos códigos.

### 3.4 · `medical_specialty`

Aproximadamente un **49%** de valores ausentes.

Con casi la mitad del dataset sin información, imputar la moda distorsionaría por
completo la distribución de la variable: la especialidad más frecuente pasaría a
representar a la mitad de los pacientes sin ningún fundamento. Se conserva la
información de ausencia mediante la categoría `Unknown`.

In [5]:
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

print(f'NaN restantes en medical_specialty: {df["medical_specialty"].isna().sum()}')

NaN restantes en medical_specialty: 0


### 3.5 · `payer_code`

Aproximadamente un **40%** de valores ausentes. Se aplica exactamente el mismo
razonamiento que en `medical_specialty`.

Además, en una variable administrativa como el código de pagador, que el dato no
conste puede reflejar un tipo concreto de circuito asistencial, lo que refuerza la
decisión de tratarlo como una categoría propia y no como un hueco.

In [6]:
df['payer_code'] = df['payer_code'].fillna('Unknown')

print(f'NaN restantes en payer_code: {df["payer_code"].isna().sum()}')

NaN restantes en payer_code: 0


### 3.6 · `max_glu_serum` y `A1Cresult`

Estas dos variables aparecen con un porcentaje altísimo de `NaN`, pero —como se
documentó en el notebook 01— **no son datos perdidos**.

En el CSV original su valor es la cadena literal `None` (96.420 y 84.748 filas), que
Pandas convierte a `NaN` al leer porque forma parte de su lista por defecto de valores
nulos. El dataset original sí registraba el valor, y ese valor significa *"la prueba
no se realizó"*.

Por tanto no se imputa nada: simplemente se **recupera el significado original** con
una categoría explícita, preservando la distinción entre un paciente con resultado
registrado y un paciente sin medición disponible.

Esta distinción es clínicamente relevante: que un facultativo decida **no** solicitar
una hemoglobina glicosilada es en sí mismo un dato sobre el seguimiento del paciente,
y podría estar relacionado con su riesgo de readmisión.

In [7]:
df[['max_glu_serum', 'A1Cresult']] = df[['max_glu_serum', 'A1Cresult']].fillna('Not_Measured')

print(df[['max_glu_serum', 'A1Cresult']].isna().sum())

max_glu_serum    0
A1Cresult        0
dtype: int64


### 3.7 · Comprobación

Tras aplicar las decisiones anteriores, los únicos `NaN` que deben quedar en el
dataset son los de las tres variables de diagnóstico.

In [8]:
nan_restantes = df.isna().sum()

print(nan_restantes[nan_restantes > 0])

diag_1      21
diag_2     358
diag_3    1423
dtype: int64


In [9]:
# Comprobación de que 'Unknown' se ha incorporado como una categoría más
print(df['payer_code'].value_counts(), end='\n\n')
print(df['medical_specialty'].value_counts().head(10))

payer_code
Unknown    40256
MC         32439
HM          6274
SP          5007
BC          4655
MD          3532
CP          2533
UN          2448
CM          1937
OG          1033
PO           592
DM           549
CH           146
WC           135
OT            95
MP            79
SI            55
FR             1
Name: count, dtype: int64

medical_specialty
Unknown                       49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64


## 4 · Variables casi constantes

Una variable en la que prácticamente todos los pacientes comparten el mismo valor
apenas distingue unos casos de otros, y por tanto difícilmente puede ayudar a explicar
la readmisión.

Se calcula, para cada variable, **el porcentaje que representa su categoría más frecuente**
(la *dominancia*), se ordenan de mayor a menor y se marcan aquellas en las que una sola
categoría supera el **99%** de los registros. Para esas, se muestran los recuentos absolutos,
que son los que permiten decidir.

> El umbral del 99% se usa como **detector**, no como regla automática de eliminación.
> Un porcentaje muy alto puede convivir con cientos de casos informativos, y esos casos
> pueden ser justamente los interesantes. La decisión final se toma mirando el número
> absoluto de observaciones distintas de la categoría dominante.

In [10]:
# Dominancia de cada variable: % de registros que concentra su categoría más frecuente.
dominancia = {}

for column in df.columns:
    dominancia[column] = df[column].value_counts(normalize=True).max() * 100

# De diccionario a Series, ordenada de mayor a menor dominancia
dominancia = pd.Series(dominancia).sort_values(ascending=False)

print('Dominancia por variable (%):')
print(dominancia.round(2).to_string(), end='\n\n')

# Detector: variables cuya categoría dominante supera el 99%
columnas_casi_constantes = list(dominancia[dominancia > 99].index)
print(f'Variables con una categoría > 99%: {len(columnas_casi_constantes)}')
print(columnas_casi_constantes, end='\n\n')

# Recuentos absolutos de esas variables, que son los que permiten decidir
for column in columnas_casi_constantes:
    print(df[column].value_counts(), end='\n\n')

Dominancia por variable (%):


metformin-pioglitazone      100.00
glimepiride-pioglitazone    100.00
acetohexamide               100.00
metformin-rosiglitazone     100.00
troglitazone                100.00
glipizide-metformin          99.99
tolbutamide                  99.98
miglitol                     99.96
tolazamide                   99.96
chlorpropamide               99.92
acarbose                     99.70
nateglinide                  99.31
glyburide-metformin          99.31
repaglinide                  98.49
glimepiride                  94.90
max_glu_serum                94.75
rosiglitazone                93.75
pioglitazone                 92.80
glyburide                    89.53
number_emergency             88.81
glipizide                    87.53
number_outpatient            83.55
A1Cresult                    83.28
metformin                    80.36
diabetesMed                  77.00
race                         74.78
number_inpatient             66.46
discharge_disposition_id     59.19
admission_source_id

### Decisión

Al revisar los recuentos absolutos aparecen dos grupos claramente distintos:

| Grupo | Observaciones fuera de la categoría dominante | Decisión |
|---|---|---|
| `chlorpropamide`, `acetohexamide`, `tolbutamide`, `miglitol`, `troglitazone`, `tolazamide`, `glipizide-metformin`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone` | Desde 1 hasta unas decenas | **Eliminar** |
| `nateglinide`, `acarbose`, `glyburide-metformin` | Varios cientos | **Conservar** |

Las del primer grupo son fármacos prácticamente no utilizados en esta cohorte: con
tan pocos casos no es posible extraer ningún patrón fiable y solo añaden ruido y
dimensionalidad.

Las del segundo grupo superan el umbral del 99%, pero contienen **cientos de pacientes**
tratados con ese fármaco. Es una cantidad suficiente como para que un modelo pueda
encontrar señal, así que eliminarlas por aplicar el umbral de forma mecánica supondría
descartar información potencialmente útil.

In [11]:
columnas_a_eliminar = [
    'chlorpropamide',
    'acetohexamide',
    'tolbutamide',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

df = df.drop(columns=columnas_a_eliminar)

print(f'Dimensiones tras eliminar las variables casi constantes: {df.shape}')

Dimensiones tras eliminar las variables casi constantes: (101766, 35)


## 5 · Conversión de tipos

El tipo que Pandas asigna a una columna refleja **cómo está escrita**, no **qué
representa**. Varias variables almacenadas como enteros son en realidad códigos de
categoría.

`admission_type_id`, `discharge_disposition_id` y `admission_source_id` contienen
números, pero esos números son etiquetas: el tipo de ingreso 6 no es "el triple" que
el 2, ni el destino al alta 3 está "entre" el 2 y el 4. Tratarlos como magnitudes
llevaría a calcular medias sin sentido y a que un modelo asumiera un orden inexistente.

Lo mismo ocurre con los códigos de diagnóstico: el código 700 no es clínicamente
"mayor" que el 200.

In [12]:
df.dtypes

race                        object
gender                      object
age                         object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
glimepiride                 object
glipizide                   object
glyburide                   object
pioglitazone                object
rosiglitazone       

In [13]:
columnas_categoricas_numericas = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

df[columnas_categoricas_numericas] = df[columnas_categoricas_numericas].astype('category')

df[['diag_1', 'diag_2', 'diag_3']] = df[['diag_1', 'diag_2', 'diag_3']].astype('category')

# Solo las seis columnas convertidas; el resto no ha cambiado respecto a la celda anterior
df[columnas_categoricas_numericas + ['diag_1', 'diag_2', 'diag_3']].dtypes

admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
diag_1                      category
diag_2                      category
diag_3                      category
dtype: object

> **Limitación del formato CSV:** los `dtype` no se guardan en el fichero. Estas
> conversiones se pierden al exportar y las tres variables `*_id` volverán a leerse
> como `int64`. Por eso el notebook de EDA las vuelve a declarar como categóricas nada
> más cargar los datos.

## 6 · Verificación de las categorías

Revisión final de los valores de todas las variables categóricas, ya con las
transformaciones aplicadas, para confirmar que no se han introducido categorías
inesperadas ni duplicadas.

In [14]:
columnas_categoricas = df.select_dtypes(include=['object', 'category']).columns

for column in columnas_categoricas:
    print(f'\n{column}:')
    print(df[column].unique())


race:
['Caucasian' 'AfricanAmerican' 'Unknown' 'Other' 'Asian' 'Hispanic']

gender:
['Female' 'Male']

age:


['[0-10)' '[10-20)' '[20-30)' '[30-40)' '[40-50)' '[50-60)' '[60-70)'
 '[70-80)' '[80-90)' '[90-100)']

admission_type_id:
[6, 1, 2, 3, 4, 5, 8, 7]
Categories (8, int64): [1, 2, 3, 4, 5, 6, 7, 8]

discharge_disposition_id:
[25, 1, 3, 6, 2, ..., 15, 24, 28, 19, 27]
Length: 26
Categories (26, int64): [1, 2, 3, 4, ..., 24, 25, 27, 28]

admission_source_id:
[1, 7, 2, 4, 5, ..., 10, 22, 11, 25, 13]
Length: 17
Categories (17, int64): [1, 2, 3, 4, ..., 17, 20, 22, 25]

payer_code:
['Unknown' 'MC' 'MD' 'HM' 'UN' 'BC' 'SP' 'CP' 'SI' 'DM' 'CM' 'CH' 'PO'
 'WC' 'OT' 'OG' 'MP' 'FR']

medical_specialty:
['Pediatrics-Endocrinology' 'Unknown' 'InternalMedicine'
 'Family/GeneralPractice' 'Cardiology' 'Surgery-General' 'Orthopedics'
 'Gastroenterology' 'Surgery-Cardiovascular/Thoracic' 'Nephrology'
 'Orthopedics-Reconstructive' 'Psychiatry' 'Emergency/Trauma'
 'Pulmonology' 'Surgery-Neuro' 'Obsterics&Gynecology-GynecologicOnco'
 'ObstetricsandGynecology' 'Pediatrics' 'Hematology/Oncology'
 'Otolaryngolo

['No' 'Steady' 'Down' 'Up']

glipizide:
['No' 'Steady' 'Up' 'Down']

glyburide:
['No' 'Steady' 'Up' 'Down']

pioglitazone:
['No' 'Steady' 'Up' 'Down']

rosiglitazone:
['No' 'Steady' 'Up' 'Down']

acarbose:
['No' 'Steady' 'Up' 'Down']

insulin:
['No' 'Up' 'Steady' 'Down']

glyburide-metformin:
['No' 'Steady' 'Down' 'Up']

change:
['No' 'Ch']

diabetesMed:
['No' 'Yes']

readmitted:
['NO' '>30' '<30']


## 7 · Registros duplicados

Las filas exactamente idénticas no aportan información y alterarían las frecuencias de
las variables y el peso de esos casos en el modelo.

In [15]:
duplicados = df.duplicated().sum()

print(f'Número de filas duplicadas: {duplicados}')

Número de filas duplicadas: 0


## 8 · Rangos de las variables numéricas

Comprobación de que las transformaciones no han alterado las magnitudes y de que sus
rangos siguen siendo plausibles.

In [16]:
columnas_numericas = df.select_dtypes(include=['int64', 'float64']).columns

for column in columnas_numericas:
    print(f'{column}: min={df[column].min()}, max={df[column].max()}')

time_in_hospital: min=1, max=14
num_lab_procedures: min=1, max=132
num_procedures: min=0, max=6
num_medications: min=1, max=81
number_outpatient: min=0, max=42
number_emergency: min=0, max=76
number_inpatient: min=0, max=21
number_diagnoses: min=1, max=16


Todos los rangos son clínicamente plausibles: estancias de 1 a 14 días, hasta 132
pruebas de laboratorio, hasta 81 medicaciones y de 1 a 16 diagnósticos.

**No se elimina ningún valor extremo.** En variables como `number_inpatient`,
`number_emergency` o `number_outpatient`, un valor alto no es un error de medida: es un
paciente que realmente ha ingresado muchas veces. Y ese perfil es, precisamente, el que
mayor riesgo de readmisión puede presentar. Eliminarlo sería borrar la señal que el
proyecto quiere detectar.

## 9 · Exportación del dataset limpio

El resultado se guarda en `data/processed/`, dejando intacto el dataset original.

In [17]:
df.to_csv(PROC / 'diabetic_data_clean.csv', index=False)

print(f'Guardado en: {PROC / "diabetic_data_clean.csv"}')
print(f'Dimensiones finales: {df.shape}')

Guardado en: D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1\data\processed\diabetic_data_clean.csv
Dimensiones finales: (101766, 35)


---

## 10 · Conclusiones de la limpieza

### Resultado

| | Antes | Después |
|---|---|---|
| Filas | 101.766 | **101.766** |
| Columnas | 50 | **35** |
| Variables con valores ausentes | 10 | **3** (los diagnósticos, intencionadamente) |
| Filas duplicadas | 0 | 0 |

**No se ha eliminado ninguna fila.** Toda la reducción proviene de columnas sin
capacidad informativa. Las diez variables con ausencias de partida eran las siete con `?`
en el original, las dos con `None` (`max_glu_serum`, `A1Cresult`) y `gender`.

### Las 15 columnas eliminadas

| Motivo | Columnas |
|---|---|
| Identificadores (2) | `encounter_id`, `patient_nbr` |
| Información insuficiente (1) | `weight` (~97% desconocido) |
| Constantes (2) | `examide`, `citoglipton` |
| Casi constantes con muy pocos casos (10) | `chlorpropamide`, `acetohexamide`, `tolbutamide`, `miglitol`, `troglitazone`, `tolazamide`, `glipizide-metformin`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone` |

### Criterios aplicados

1. **Representar la ausencia antes que inventar el valor.** Solo se ha imputado con la
   moda en `gender`, donde el número de registros afectados (3) era despreciable. En el
   resto, la ausencia se ha convertido en una categoría propia (`Unknown`,
   `Not_Measured`) o se ha conservado como `NaN`.

2. **Distinguir "dato ausente" de "dato que dice que no hay medición".** En
   `max_glu_serum` y `A1Cresult` la ausencia de prueba es información clínica, no un
   hueco.

3. **Los umbrales automáticos detectan, no deciden.** El criterio del 99% señaló 13
   variables; se conservaron 3 tras comprobar que contenían cientos de casos
   informativos.

4. **La naturaleza de la variable manda sobre el tipo de Pandas.** Los códigos
   administrativos y de diagnóstico se han declarado como categorías, no como números.

5. **No se han eliminado valores extremos.** Un outlier estadístico no es
   necesariamente un error clínico.

### Cuestiones abiertas para las fases siguientes

- **Duplicidad de pacientes:** 101.766 ingresos corresponden a 71.518 pacientes. Habrá
  que tenerlo en cuenta al separar entrenamiento y test.
- **Alta cardinalidad de los diagnósticos:** `diag_1`, `diag_2` y `diag_3` tienen entre
  700 y 800 categorías. Requerirán agrupación antes del modelado.
- **Desbalance de la variable objetivo:** se cuantificará en el notebook de EDA.

### Siguiente paso

Análisis exploratorio en **`03_EDA.ipynb`**, partiendo de `diabetic_data_clean.csv`.